# MedleyVox One Click Gradio

แยกเสียงนักร้องหลายคนในเพลงเดียว กดรันเซลล์ด้านล่างเซลล์เดียว แล้วรอจนขึ้นลิงก์ Gradio

In [ ]:
# @title กดรันครั้งเดียวเพื่อเปิด MedleyVox (แยกเสียงนักร้องหลายคน)
# @markdown ต้องเลือกเครื่องแบบ GPU ก่อน: เมนู Runtime > Change runtime type > T4 GPU
language = "th_TH" # @param ["th_TH", "en_US"]

from IPython.display import clear_output
from pathlib import Path
import json, os

repo_path = Path('/content/MedleyVox-Inference-WebUI')
py = '/content/mvenv/bin/python'

%cd /content
if repo_path.exists():
    %cd /content/MedleyVox-Inference-WebUI
    !git checkout -q -- webui.py inference.py
    !git pull -q
else:
    !git clone -q https://github.com/SUC-DriverOld/MedleyVox-Inference-WebUI
    %cd /content/MedleyVox-Inference-WebUI

import subprocess, sys

def run(step, cmd):
    # รันทีละขั้น ถ้าพังให้หยุดและแสดงสาเหตุ (ไม่ลบข้อความทิ้ง)
    print(f'▶ {step} ...')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-3000:])
        raise SystemExit(f'❌ ขั้น "{step}" ไม่สำเร็จ — ส่งข้อความด้านบนให้คนดูแลได้เลย')

if not Path('/content/mvenv/.ok').exists():
    print('กำลังติดตั้งของจำเป็น (ครั้งแรกใช้เวลาประมาณ 10-15 นาที) ...')
    # fairseq ติดตั้งบน Python ของ Colab (3.12) ไม่ได้ จึงแยกใช้ Python 3.10
    run('ติดตั้งตัวช่วย uv', f'{sys.executable} -m pip install -q uv')
    run('สร้าง Python 3.10', f'rm -rf /content/mvenv && {sys.executable} -m uv venv --seed -p 3.10 /content/mvenv')
    run('เตรียม pip', f'{py} -m pip install -q "pip<24.1" setuptools wheel cython numpy')
    run('ติดตั้ง torch', f'{py} -m pip install -q "torch<2.6.0" torchaudio --index-url https://download.pytorch.org/whl/cu124')
    run('ติดตั้งส่วนประกอบ MedleyVox (นานสุด)', f'{py} -m pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124')
    Path('/content/mvenv/.ok').touch()
else:
    print('ติดตั้งไว้แล้ว ข้ามขั้นติดตั้ง')

# librosa รุ่นที่ MedleyVox ใช้ ต้องการ pkg_resources ซึ่ง setuptools รุ่นใหม่ตัดออกแล้ว -> ตรึงรุ่นเก่า (ทำทุกครั้ง ใช้เวลาไม่กี่วินาที)
run('ตรึง setuptools ให้ librosa ใช้ได้', f'{py} -m pip install -q "setuptools<81"')
run('ตรวจว่าติดตั้งครบ', f'{py} -c "import pkg_resources, torch, fairseq, asteroid, librosa, gradio; print(torch.cuda.is_available())"')

print('กำลังดาวน์โหลดโมเดล ...')
models = {
    'vocals_238': 'vocals%20238',
    'multi_singing_librispeech_138': 'multi_singing_librispeech_138',
    'singing_librispeech_ft_iSRNet': 'singing_librispeech_ft_iSRNet',
}
for name, remote in models.items():
    folder = repo_path / 'checkpoints' / name
    folder.mkdir(parents=True, exist_ok=True)
    for f in ['vocals.pth', 'vocals.json']:
        if not (folder / f).exists():
            !wget -q -O "{folder}/{f}" "https://huggingface.co/Cyru5/MedleyVox/resolve/main/{remote}/{f}"
(repo_path / 'pretrained').mkdir(exist_ok=True)
if not (repo_path / 'pretrained' / 'xlsr_53_56k.pt').exists():
    !wget -q -O pretrained/xlsr_53_56k.pt https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt
for d in ['inputs', 'results']:
    (repo_path / d).mkdir(exist_ok=True)

# ไฟล์ภาษาไทย (ต้นฉบับไม่มี จึงใส่มาในโน้ตบุ๊กนี้)
th_TH = {
 "Input audio files to inference": "แยกเสียงจากไฟล์เสียง",
 "Input folder to inference": "แยกเสียงจากโฟลเดอร์",
 "Please upload at least one audio file!": "กรุณาอัปโหลดไฟล์เสียงอย่างน้อย 1 ไฟล์",
 "Please download the pretrained model xlsr_53_56k.pt from https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt and put it in the 'pretrained' folder!": "กรุณาดาวน์โหลดโมเดล xlsr_53_56k.pt จาก https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt แล้วนำไปไว้ในโฟลเดอร์ 'pretrained'",
 "Please select a model!": "กรุณาเลือกโมเดล",
 "Please select an output folder!": "กรุณาเลือกโฟลเดอร์ที่จะเก็บผลลัพธ์",
 "Please select an input folder!": "กรุณาเลือกโฟลเดอร์ไฟล์ต้นฉบับ",
 "Inference process finished.": "แยกเสียงเสร็จแล้ว",
 "Inference process stopped by user.": "หยุดการแยกเสียงแล้ว",
 "No active inference process.": "ไม่มีงานแยกเสียงที่กำลังทำอยู่",
 "Medley Vox is a [dataset for testing algorithms for separating multiple singers](https://arxiv.org/pdf/2211.07302) within a single music track. Also, the [authors of Medley Vox](https://github.com/jeonchangbin49/MedleyVox) proposed a neural network architecture for separating singers. However, unfortunately, they did not publish the weights. Later, their training process was [repeated by Cyru5](https://huggingface.co/Cyru5/MedleyVox/tree/main), who trained several models and published the weights in the public domain. Now this WebUI is created to use the trained models and weights for inference. Here are some precautions:<br>1. Put the [downloaded models](https://huggingface.co/Cyru5/MedleyVox) in the 'checkpoints' folder in folder format, with each model folder containing a model file (.pth) and its corresponding configuration file (.json).<br>2. If you use overlapadd and the choice of model is 'w2v' or 'w2v_chunk', you need to download the pretrained model [xlsr_53_56k.pt](https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt) and put it in the 'pretrained' folder.<br>3. At present, the audio output sampling rate supported by the model is 24000kHz and cannot be changed. To solve this, you can use [AudioSR](https://github.com/haoheliu/versatile_audio_super_resolution), [Apollo](https://github.com/JusperLee/Apollo), or [Music Source Separation Training](https://github.com/ZFTurbo/Music-Source-Separation-Training) for audio super-resolution.<br>4. When using WebUI on cloud platforms or Colab, please place the audio to be processed in the 'inputs' folder, and the processing results will be stored in the 'results' folder. The 'Select folder' and 'Open folder' buttons are invalid in the cloud.": "Medley Vox คือ[ชุดข้อมูลสำหรับทดสอบวิธีแยกเสียงนักร้องหลายคน](https://arxiv.org/pdf/2211.07302)ที่ร้องอยู่ในเพลงเดียวกัน [ทีมผู้พัฒนา Medley Vox](https://github.com/jeonchangbin49/MedleyVox) ได้เสนอโครงข่ายประสาทเทียมสำหรับแยกเสียงนักร้องไว้ แต่ไม่ได้เผยแพร่ไฟล์โมเดลที่ฝึกแล้ว ต่อมา [Cyru5 ได้ฝึกโมเดลตามวิธีเดิมซ้ำ](https://huggingface.co/Cyru5/MedleyVox/tree/main) และเผยแพร่ให้ใช้ได้ฟรี หน้าเว็บนี้สร้างขึ้นเพื่อใช้โมเดลเหล่านั้นแยกเสียง ข้อควรทราบมีดังนี้<br>1. นำ[โมเดลที่ดาวน์โหลดมา](https://huggingface.co/Cyru5/MedleyVox)ไปไว้ในโฟลเดอร์ 'checkpoints' โดยแต่ละโมเดลอยู่ในโฟลเดอร์ของตัวเอง ภายในมีไฟล์โมเดล (.pth) และไฟล์ตั้งค่า (.json) คู่กัน<br>2. หากใช้ overlapadd แบบ 'w2v' หรือ 'w2v_chunk' ต้องดาวน์โหลดโมเดล [xlsr_53_56k.pt](https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt) ไปไว้ในโฟลเดอร์ 'pretrained'<br>3. ขณะนี้โมเดลส่งออกเสียงที่อัตราสุ่ม 24000 Hz เท่านั้น เปลี่ยนไม่ได้ หากต้องการเสียงคมชัดขึ้น ใช้ [AudioSR](https://github.com/haoheliu/versatile_audio_super_resolution), [Apollo](https://github.com/JusperLee/Apollo) หรือ [Music Source Separation Training](https://github.com/ZFTurbo/Music-Source-Separation-Training) ช่วยเพิ่มความละเอียดเสียงได้<br>4. หากใช้บนคลาวด์หรือ Colab ให้นำไฟล์เสียงไปไว้ในโฟลเดอร์ 'inputs' แล้วผลลัพธ์จะอยู่ในโฟลเดอร์ 'results' ปุ่ม 'เลือกโฟลเดอร์' และ 'เปิดโฟลเดอร์' ใช้งานบนคลาวด์ไม่ได้",
 "Common options": "ตั้งค่าทั่วไป",
 "Select Model": "เลือกโมเดล",
 "Select which model you want to use.": "💡 แนะนำ: เริ่มที่ vocals_238 (คะแนนแยกเสียงดีที่สุดใน 3 ตัว) | singing_librispeech_ft_iSRNet = รุ่นต่อยอดจาก vocals_238 เพิ่มขั้นเก็บรายละเอียดเสียงแหลม ลองเมื่อผลฟังอู้อี้ | multi_singing_librispeech_138 = ฝึกกับเพลงที่มีคนร้องหลายคน ลองเมื่อร้องพร้อมกัน 3 คนขึ้นไปหรือคอรัส | ทุกตัวแยกออกมาเป็น 2 เสียงต่อครั้ง และควรใส่เสียงร้องที่แยกดนตรีออกแล้ว",
 "Use overlapadd": "วิธีแบ่งช่วงประมวลผล (overlapadd) — แนะนำ ola",
 "Use overlapadd functions, ola, ola_norm, w2v will work with ola_window_len, ola_hop_len argugments. w2v_chunk and sf_chunk is chunk-wise processing based on VAD, so you have to specify the vad_method args. If you use sf_chunk (spectral_featrues_chunk), you also need to specify spectral_features.<br>If the input is too long, it may be impossible to inference due to lack of VRAM. In that case, use 'use_overlapadd'. Among the 'use_overlapadd' options, 'ola', 'ola_norm', and 'w2v' all work well. Use w2v_chunk or sf_chunk if these fail or as desired. You can also try experimenting with 'vad_method' options spec and webrtc when using either of the '_chunk' methods. Chunking has proven to be very useful therefore it is on by default.": "💡 ตัดเพลงเป็นท่อนสั้นๆ ทยอยแยก แล้วต่อกลับ (กันการ์ดจอเต็ม) | ola ⭐ = ตัดท่อนยาวเท่ากัน ซ้อนเหลื่อมกันนิดหน่อย ใช้ได้กับแทบทุกเพลง | ola_norm = แบบ ola แต่ปรับความดังแต่ละท่อนให้เท่ากัน ลองเมื่อเพลงมีช่วงเบา-ดังต่างกันมาก | w2v = ใช้ AI ฟังว่าแต่ละท่อนเป็นเสียงคนไหน ลองเมื่อเสียง 2 คนสลับไฟล์กันกลางเพลง (ช้ากว่า) | w2v_chunk / sf_chunk = ตัดตามช่วงที่เงียบแทนตัดเท่ากัน ลองเมื่อ 3 แบบแรกไม่ได้ผล (ต้องเลือกวิธี VAD ในตั้งค่าขั้นสูง) | None = แยกทั้งเพลงทีเดียว เพลงยาวอาจค้างเพราะการ์ดจอเต็ม",
 "Use GPU": "ใช้การ์ดจอ (GPU)",
 "Use GPU for inference.": "ใช้การ์ดจอช่วยประมวลผล (เร็วกว่า)",
 "Save results in separate folders": "เก็บผลลัพธ์แยกโฟลเดอร์",
 "Save results in separate folders with the same name as the input file.": "เก็บผลลัพธ์ของแต่ละไฟล์ในโฟลเดอร์ชื่อเดียวกับไฟล์ต้นฉบับ",
 "Skip error files": "ข้ามไฟล์ที่มีปัญหา",
 "Skip error files while separating instead of stopping.": "หากไฟล์ใดมีปัญหา ให้ข้ามไปทำไฟล์ถัดไปแทนการหยุดทั้งหมด",
 "Output format": "ชนิดไฟล์ผลลัพธ์",
 "Select the output format.": "เลือกชนิดไฟล์เสียงที่ต้องการ",
 "[Click to expand] Advanced options": "[กดเพื่อเปิด] ตั้งค่าขั้นสูง",
 "VAD method": "วิธีตรวจหาช่วงที่มีเสียงร้อง (VAD)",
 "What method do you want to use for 'voice activity detection (vad) -- split chunks -- processing. Only valid when 'w2v_chunk' or 'sf_chunk' for args.use_overlapadd.": "💡 ใช้เฉพาะเมื่อเลือก w2v_chunk หรือ sf_chunk | spec ⭐ = ดูจากภาพคลื่นเสียง ใช้ได้กับเพลงทั่วไป | webrtc = ตัวจับเสียงพูดของ Google ลองเมื่อ spec ตัดท่อนผิดที่",
 "Spectral features": "ลักษณะเสียงที่ใช้เทียบ (spectral features)",
 "What spectral feature do you want to use in correlation calc in speaker assignment (only valid when using sf_chunk)": "💡 ใช้เฉพาะ sf_chunk: ใช้ลักษณะเสียงแบบไหนเทียบว่าท่อนไหนเป็นคนไหน | mfcc ⭐ = เทียบโทนเสียงโดยรวม ใช้ได้ทั่วไป | spectral_centroid = เทียบความสว่าง/แหลมของเสียง ลองเมื่อ 2 คนเสียงสูง-ต่ำต่างกันชัด",
 "OLA window length": "ความยาวช่วง OLA",
 "OLA window size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None).": "💡 ความยาวแต่ละท่อน (วินาที) ใช้กับ ola / ola_norm | 0 ⭐ = ค่ามาตรฐาน 3 วินาที (ความยาวที่โมเดลฝึกมา) | ยาวขึ้น (เช่น 5) = เสียงต่อเนื่องขึ้นแต่กินการ์ดจอมากขึ้น",
 "OLA hop length": "ระยะเลื่อน OLA",
 "OLA hop size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None).": "💡 ระยะเลื่อนไปท่อนถัดไป (วินาที) | 0 ⭐ = ค่ามาตรฐาน 0.75 วินาที (1/4 ของความยาวท่อน) | ยิ่งน้อย = ท่อนซ้อนกันมาก รอยต่อเนียนขึ้นแต่ช้าลง",
 "Wav2Vec nth layer output": "ชั้นผลลัพธ์ของ Wav2Vec",
 "Wav2Vec nth layer output, only valid when using w2v or w2v_chunk. For example: 0 1 2 3, default: 0": "💡 ใช้เฉพาะ w2v / w2v_chunk | 0 ⭐ = ค่าเริ่มต้น ปล่อยไว้ได้ | เป็นตัวเลือกสำหรับทดลอง ไม่จำเป็นต้องแก้",
 "Use EMA model": "ใช้โมเดลแบบ EMA",
 "Use EMA model or online model? Only vaind when args.ema it True (model trained with EMA).": "💡 เปิดไว้ ⭐ — ทั้ง 3 โมเดลฝึกแบบ EMA ไว้ เวอร์ชัน EMA มักให้ผลนิ่งกว่า",
 "Mix consistent output": "ปรับผลรวมให้ตรงต้นฉบับ",
 "Only valid when the model is trained with mixture_consistency loss.": "💡 เปิดไว้ ⭐ — บังคับให้ 2 เสียงที่แยกได้ รวมกันแล้วเท่ากับต้นฉบับ ช่วยไม่ให้เสียงหาย (ทั้ง 3 โมเดลรองรับ)",
 "Reorder chunks": "เรียงช่วงใหม่",
 "OLA reorder chunks. Only valid when using ola or ola_norm.": "💡 เปิดไว้ ⭐ — จับคู่ว่าแต่ละท่อนเป็นเสียงคนไหน ให้คนเดิมอยู่ไฟล์เดิมตลอดเพลง ถ้าปิด เสียง 2 คนจะสลับไฟล์กันไปมา",
 "Input audio files": "ไฟล์เสียง",
 "Input one or more audio files": "ใส่ไฟล์เสียง 1 ไฟล์ขึ้นไป",
 "Input folder path": "ที่อยู่โฟลเดอร์ต้นฉบับ",
 "Audio files in the folder will be used for inference.": "ไฟล์เสียงทั้งหมดในโฟลเดอร์นี้จะถูกนำไปแยกเสียง",
 "Select folder": "เลือกโฟลเดอร์",
 "Open folder": "เปิดโฟลเดอร์",
 "Output folder path": "ที่อยู่โฟลเดอร์ผลลัพธ์",
 "Audio files will be saved in this folder.": "ไฟล์ที่แยกเสร็จจะเก็บไว้ในโฟลเดอร์นี้",
 "Output message": "ข้อความแจ้ง",
 "Forced stop inference": "หยุดการแยกเสียง",
 "Loading model...": "กำลังโหลดโมเดล...",
 "Reading audio file...": "กำลังอ่านไฟล์เสียง",
 "Separating": "กำลังแยกเสียง",
 "Saving files...": "กำลังบันทึกไฟล์",
 "file": "ไฟล์",
 "Elapsed": "ผ่านไป",
 "About": "เหลืออีกประมาณ",
 "left": "",
 "Estimating time left...": "กำลังคำนวณเวลาที่เหลือ...",
 "Chunks done": "แยกเสร็จแล้ว (ท่อน)",
 "this method cannot estimate time left": "วิธีแบ่งช่วงแบบนี้คำนวณเวลาที่เหลือไม่ได้",
 "Another separation is still running. Please wait or press stop.": "ยังมีงานแยกเสียงทำอยู่ กรุณารอให้เสร็จ หรือกดหยุดก่อน",
 "Took": "ใช้เวลา",
 "Files are in": "ไฟล์อยู่ที่",
 "Finished with errors": "เสร็จแล้ว แต่มีไฟล์ที่แยกไม่สำเร็จ",
 "Details in": "ดูรายละเอียดที่",
 "Separation failed": "แยกเสียงไม่สำเร็จ",
 "Close MedleyVox program": "ปิดโปรแกรม MedleyVox",
 "MedleyVox is closed. You can close this browser tab.": "ปิดโปรแกรมแล้ว ปิดแท็บนี้ได้เลย",
 "Finishing...": "กำลังจบงาน"
}
(repo_path / 'utils' / 'locale' / 'th_TH.json').write_text(json.dumps(th_TH, ensure_ascii=False, indent=4), encoding='utf-8')
# ใช้ webui.py / inference.py ฉบับปรับปรุง: แถบความคืบหน้า % + เวลาที่เหลือ, ปุ่มหยุดที่หยุดได้จริง, โมเดลเริ่มต้น vocals_238
patched_files = {'webui.py': '__author__ = "Sucial: https://github.com/SUC-DriverOld"\n__version__ = "1.0.0"\n\nimport os\nimport sys\nimport shutil\nimport glob\nimport gradio as gr\nimport multiprocessing\nimport subprocess\nimport json\nimport time\nimport tkinter as tk\nfrom tkinter import filedialog\nfrom utils import I18nAuto\n\n\nos.makedirs("checkpoints", exist_ok=True)\nos.makedirs("pretrained", exist_ok=True)\nMODEL_DIR = "checkpoints"\nPRETRAINED_MODEL_DIR = "pretrained"\nTEMP_DIR = "temp"\nPYTHON = sys.executable\nmodel_list = [m for m in os.listdir(MODEL_DIR) if os.path.isdir(os.path.join(MODEL_DIR, m))]\ni18n = I18nAuto(language=os.environ.get("LANGUAGE", "auto"))\n\ndef select_folder():\n    root = tk.Tk()\n    root.withdraw()\n    root.attributes(\'-topmost\', True)\n    selected_dir = filedialog.askdirectory()\n    root.destroy()\n    return selected_dir\n\ndef open_folder(folder):\n    os.makedirs(folder, exist_ok=True)\n    absolute_path = os.path.abspath(folder)\n    if os.name == "nt":\n        os.system(f"explorer {absolute_path}")\n    elif os.name == "posix":\n        os.system(f"xdg-open {absolute_path}")\n    else:\n        os.system(f"open {absolute_path}")\n\ndef change_to_audio_infer():\n    return (gr.Button(i18n("Input audio files to inference"), variant="primary", visible=True),\n            gr.Button(i18n("Input folder to inference"), variant="primary", visible=False))\n\ndef change_to_folder_infer():\n    return (gr.Button(i18n("Input audio files to inference"), variant="primary", visible=False),\n            gr.Button(i18n("Input folder to inference"), variant="primary", visible=True))\n\ndef inference_audio_fn(model_name, use_overlapadd, use_gpu, separate_storage, output_format, vad_method, spectral_features, ola_window_len, ola_hop_len, w2v_nth_layer_output, use_ema_model, mix_consistent_out, reorder_chunks, skip_error, audio_input, store_dir):\n    if not audio_input:\n        yield i18n("Please upload at least one audio file!")\n        return\n    if os.path.exists(TEMP_DIR):\n        shutil.rmtree(TEMP_DIR)\n    os.makedirs(TEMP_DIR, exist_ok=True)\n    for audio in audio_input:\n        shutil.copy(audio, TEMP_DIR)\n    folder_input = TEMP_DIR\n    try:\n        yield from inference_folder_fn(model_name, use_overlapadd, use_gpu, separate_storage, output_format, vad_method, spectral_features, ola_window_len, ola_hop_len, w2v_nth_layer_output, use_ema_model, mix_consistent_out, reorder_chunks, skip_error, folder_input, store_dir)\n    finally:\n        shutil.rmtree(TEMP_DIR, ignore_errors=True)\n\ndef inference_folder_fn(model_name, use_overlapadd, use_gpu, separate_storage, output_format, vad_method, spectral_features, ola_window_len, ola_hop_len, w2v_nth_layer_output, use_ema_model, mix_consistent_out, reorder_chunks, skip_error, folder_input, store_dir):\n    if (use_overlapadd == "w2v" or use_overlapadd == "w2v_chunk") and not os.path.exists(os.path.join(PRETRAINED_MODEL_DIR, "xlsr_53_56k.pt")):\n        yield i18n("Please download the pretrained model xlsr_53_56k.pt from https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt and put it in the \'pretrained\' folder!")\n        return\n    if not model_name:\n        yield i18n("Please select a model!")\n        return\n    if not store_dir:\n        yield i18n("Please select an output folder!")\n        return\n    if not os.path.exists(folder_input):\n        yield i18n("Please select an input folder!")\n        return\n    yield from run_with_progress(model_name, use_overlapadd, use_gpu, separate_storage, output_format, vad_method, spectral_features, ola_window_len, ola_hop_len, w2v_nth_layer_output, use_ema_model, mix_consistent_out, reorder_chunks, skip_error, folder_input, store_dir)\n\n\n# ---- ความคืบหน้า: เปิดตัวแยกเสียงแบบไม่มีหน้าต่าง cmd แล้วอ่านไฟล์ความคืบหน้าทุก 1 วินาที ----\nCURRENT_PROCESS = None\nSTOPPED = [False]\nPROGRESS_FILE = os.path.abspath(os.path.join("cache", "progress.json"))\nSTAGE_TEXT = {\n    "loading": "Loading model...",\n    "reading": "Reading audio file...",\n    "separating": "Separating",\n    "saving": "Saving files...",\n    "finished": "Finishing...",\n}\n\n\ndef fmt_time(sec):\n    sec = int(max(sec, 0))\n    return f"{sec // 3600}:{sec % 3600 // 60:02d}:{sec % 60:02d}" if sec >= 3600 else f"{sec // 60}:{sec % 60:02d}"\n\n\ndef progress_text(p, start):\n    elapsed = time.time() - start\n    if not p:\n        return f"⏳ {i18n(\'Loading model...\')}\\n{i18n(\'Elapsed\')} {fmt_time(elapsed)}"\n    stage = p.get("stage", "loading")\n    head = f"⏳ {i18n(STAGE_TEXT.get(stage, stage))}"\n    if p.get("file"):\n        head += f" {p[\'file\']} ({i18n(\'file\')} {p.get(\'file_idx\', 0)}/{p.get(\'n_files\', 0)})"\n    lines = [head]\n    total, done = p.get("total"), p.get("done", 0)\n    if stage == "separating" and total:\n        frac = min(done / total, 0.99)\n        filled = int(frac * 20)\n        lines.append(f"{\'█\' * filled}{\'░\' * (20 - filled)} {frac * 100:.0f}%")\n        file_elapsed = time.time() - p.get("file_start", time.time())\n        if frac >= 0.03:\n            remain = f" · {i18n(\'About\')} {fmt_time(file_elapsed / frac - file_elapsed)} {i18n(\'left\')}"\n        else:\n            remain = f" · {i18n(\'Estimating time left...\')}"\n        lines.append(f"{i18n(\'Elapsed\')} {fmt_time(elapsed)}{remain}")\n    elif stage == "separating":\n        lines.append(f"{i18n(\'Chunks done\')}: {done}")\n        lines.append(f"{i18n(\'Elapsed\')} {fmt_time(elapsed)} ({i18n(\'this method cannot estimate time left\')})")\n    else:\n        lines.append(f"{i18n(\'Elapsed\')} {fmt_time(elapsed)}")\n    return "\\n".join(lines)\n\n\ndef read_progress():\n    try:\n        with open(PROGRESS_FILE, encoding="utf-8") as f:\n            return json.load(f)\n    except Exception:\n        return None\n\n\ndef run_with_progress(*params):\n    global CURRENT_PROCESS\n    if CURRENT_PROCESS and CURRENT_PROCESS.poll() is None:\n        yield i18n("Another separation is still running. Please wait or press stop.")\n        return\n    store_dir = params[-1]\n    cmd = build_command(*params)\n    print(" ".join(cmd))\n    if os.path.exists(PROGRESS_FILE):\n        os.remove(PROGRESS_FILE)\n    env = dict(os.environ, MEDLEYVOX_PROGRESS=PROGRESS_FILE, PYTHONIOENCODING="utf-8")\n    log_path = os.path.join("cache", "inference.log")\n    log = open(log_path, "w", encoding="utf-8")\n    start = time.time()\n    STOPPED[0] = False\n    CURRENT_PROCESS = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env,\n                                       creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))\n    while CURRENT_PROCESS.poll() is None:\n        yield progress_text(read_progress(), start)\n        time.sleep(1)\n    log.close()\n    CURRENT_PROCESS = None\n    p = read_progress() or {}\n    took = fmt_time(time.time() - start)\n    if STOPPED[0]:\n        yield f"⏹ {i18n(\'Inference process stopped by user.\')}"\n    elif p.get("stage") == "finished" and not p.get("errors"):\n        yield f"✅ {i18n(\'Inference process finished.\')} {i18n(\'Took\')} {took}\\n{i18n(\'Files are in\')}: {os.path.abspath(store_dir)}"\n    elif p.get("stage") == "finished":\n        yield f"⚠️ {i18n(\'Finished with errors\')}: {\', \'.join(p[\'errors\'])} ({i18n(\'Took\')} {took})\\n{i18n(\'Details in\')} cache/inference.log"\n    else:\n        try:\n            with open(log_path, encoding="utf-8", errors="replace") as f:\n                tail = "".join(f.readlines()[-8:])\n        except Exception:\n            tail = ""\n        yield f"❌ {i18n(\'Separation failed\')}\\n{tail}"\n\ndef build_command(model_name, use_overlapadd, use_gpu, separate_storage, output_format, vad_method, spectral_features, ola_window_len, ola_hop_len, w2v_nth_layer_output, use_ema_model, mix_consistent_out, reorder_chunks, skip_error, folder_input, store_dir):\n    model_file = os.path.basename(glob.glob(os.path.join(MODEL_DIR, model_name, "*.pth"))[0])\n    yn = lambda v: "y" if v else "n"\n    cmd = [PYTHON, "inference.py", "--target", model_file.replace(".pth", ""), "--exp_name", model_name,\n           "--model_dir", MODEL_DIR, "--use_gpu", yn(use_gpu)]\n    if use_overlapadd != "None":\n        cmd += ["--use_overlapadd", use_overlapadd]\n    cmd += ["--vad_method", vad_method, "--spectral_features", spectral_features, "--w2v_ckpt_dir", PRETRAINED_MODEL_DIR,\n            "--w2v_nth_layer_output", *str(w2v_nth_layer_output).split()]\n    if ola_window_len != 0:\n        cmd += ["--ola_window_len", str(ola_window_len)]\n    if ola_hop_len != 0:\n        cmd += ["--ola_hop_len", str(ola_hop_len)]\n    cmd += ["--use_ema_model", yn(use_ema_model), "--mix_consistent_out", yn(mix_consistent_out),\n            "--reorder_chunks", yn(reorder_chunks), "--skip_error", yn(skip_error),\n            "--separate_storage", yn(separate_storage), "--output_format", output_format,\n            "--inference_data_dir", folder_input, "--results_save_dir", store_dir]\n    return cmd\n\n\ndef stop_inference_fn():\n    if CURRENT_PROCESS and CURRENT_PROCESS.poll() is None:\n        STOPPED[0] = True\n        CURRENT_PROCESS.kill()\n        return i18n("Inference process stopped by user.")\n    return i18n("No active inference process.")\n\n\ndef quit_app_fn():\n    # ปิดโปรแกรมทั้งหมด (จำเป็นเมื่อเปิดแบบไม่มีหน้าต่าง cmd)\n    if CURRENT_PROCESS and CURRENT_PROCESS.poll() is None:\n        CURRENT_PROCESS.kill()\n    import threading\n    threading.Timer(1.0, lambda: os._exit(0)).start()\n    return i18n("MedleyVox is closed. You can close this browser tab.")\n\ndef webui():\n    with gr.Blocks() as demo:\n        gr.Markdown(value=\'\'\'<div align="center"><font size=6><b>MedleyVox-Inference-WebUI</b></font></div>\'\'\')\n        gr.Markdown(value=i18n("Medley Vox is a [dataset for testing algorithms for separating multiple singers](https://arxiv.org/pdf/2211.07302) within a single music track. Also, the [authors of Medley Vox](https://github.com/jeonchangbin49/MedleyVox) proposed a neural network architecture for separating singers. However, unfortunately, they did not publish the weights. Later, their training process was [repeated by Cyru5](https://huggingface.co/Cyru5/MedleyVox/tree/main), who trained several models and published the weights in the public domain. Now this WebUI is created to use the trained models and weights for inference. Here are some precautions:<br>1. Put the [downloaded models](https://huggingface.co/Cyru5/MedleyVox) in the \'checkpoints\' folder in folder format, with each model folder containing a model file (.pth) and its corresponding configuration file (.json).<br>2. If you use overlapadd and the choice of model is \'w2v\' or \'w2v_chunk\', you need to download the pretrained model [xlsr_53_56k.pt](https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt) and put it in the \'pretrained\' folder.<br>3. At present, the audio output sampling rate supported by the model is 24000kHz and cannot be changed. To solve this, you can use [AudioSR](https://github.com/haoheliu/versatile_audio_super_resolution), [Apollo](https://github.com/JusperLee/Apollo), or [Music Source Separation Training](https://github.com/ZFTurbo/Music-Source-Separation-Training) for audio super-resolution.<br>4. When using WebUI on cloud platforms or Colab, please place the audio to be processed in the \'inputs\' folder, and the processing results will be stored in the \'results\' folder. The \'Select folder\' and \'Open folder\' buttons are invalid in the cloud."))\n        with gr.Accordion(open=True, label=i18n("Common options")):\n            with gr.Row():\n                with gr.Column():\n                    model_name = gr.Dropdown(label=i18n("Select Model"), info=i18n("Select which model you want to use."), choices=model_list, value="vocals_238" if "vocals_238" in model_list else None, interactive=True, scale=4)\n                    use_overlapadd = gr.Radio(label=i18n("Use overlapadd"), info=i18n("Use overlapadd functions, ola, ola_norm, w2v will work with ola_window_len, ola_hop_len argugments. w2v_chunk and sf_chunk is chunk-wise processing based on VAD, so you have to specify the vad_method args. If you use sf_chunk (spectral_featrues_chunk), you also need to specify spectral_features.<br>If the input is too long, it may be impossible to inference due to lack of VRAM. In that case, use \'use_overlapadd\'. Among the \'use_overlapadd\' options, \'ola\', \'ola_norm\', and \'w2v\' all work well. Use w2v_chunk or sf_chunk if these fail or as desired. You can also try experimenting with \'vad_method\' options spec and webrtc when using either of the \'_chunk\' methods. Chunking has proven to be very useful therefore it is on by default."), choices=["None", "ola", "ola_norm", "w2v", "w2v_chunk", "sf_chunk"], value="ola", interactive=True)\n                with gr.Column():\n                    use_gpu = gr.Checkbox(label=i18n("Use GPU"), info=i18n("Use GPU for inference."), value=True, interactive=True)\n                    separate_storage = gr.Checkbox(label=i18n("Save results in separate folders"), info=i18n("Save results in separate folders with the same name as the input file."), value=False, interactive=True)\n                    skip_error = gr.Checkbox(label=i18n("Skip error files"), info=i18n("Skip error files while separating instead of stopping."), value=True, interactive=True)\n                    output_format = gr.Radio(label=i18n("Output format"), info=i18n("Select the output format."), choices=["wav", "flac", "mp3"], value="wav", interactive=True)\n        with gr.Accordion(open=False, label=i18n("[Click to expand] Advanced options")):\n            with gr.Row():\n                with gr.Column():\n                    vad_method = gr.Radio(label=i18n("VAD method"), info=i18n("What method do you want to use for \'voice activity detection (vad) -- split chunks -- processing. Only valid when \'w2v_chunk\' or \'sf_chunk\' for args.use_overlapadd."), choices=["spec", "webrtc"], value="spec", interactive=True)\n                    spectral_features = gr.Radio(label=i18n("Spectral features"), info=i18n("What spectral feature do you want to use in correlation calc in speaker assignment (only valid when using sf_chunk)"), choices=["mfcc", "spectral_centroid"], value="mfcc", interactive=True)\n                    ola_window_len = gr.Number(label=i18n("OLA window length"), info=i18n("OLA window size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None)."), value=0, interactive=True, step=0.01)\n                    ola_hop_len = gr.Number(label=i18n("OLA hop length"), info=i18n("OLA hop size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None)."), value=0, interactive=True, step=0.01)\n                with gr.Column():\n                    w2v_nth_layer_output = gr.Textbox(label=i18n("Wav2Vec nth layer output"), info=i18n("Wav2Vec nth layer output, only valid when using w2v or w2v_chunk. For example: 0 1 2 3, default: 0"), value="0", interactive=True)\n                    use_ema_model = gr.Checkbox(label=i18n("Use EMA model"), info=i18n("Use EMA model or online model? Only vaind when args.ema it True (model trained with EMA)."), value=True, interactive=True)\n                    mix_consistent_out = gr.Checkbox(label=i18n("Mix consistent output"), info=i18n("Only valid when the model is trained with mixture_consistency loss."), value=True, interactive=True)\n                    reorder_chunks = gr.Checkbox(label=i18n("Reorder chunks"), info=i18n("OLA reorder chunks. Only valid when using ola or ola_norm."), value=True, interactive=True)\n        with gr.Row():\n            with gr.Column():\n                with gr.Tabs():\n                    with gr.TabItem(label=i18n("Input audio files")) as audio_tab:\n                        audio_input = gr.Files(label=i18n("Input one or more audio files"), type="filepath")\n                    with gr.TabItem(label=i18n("Input folder path")) as folder_tab:\n                        folder_input = gr.Textbox(label=i18n("Input folder path"), info=i18n("Audio files in the folder will be used for inference."), value="inputs/", interactive=True, scale=4)\n                        with gr.Row():\n                            select_input_dir = gr.Button(i18n("Select folder"))\n                            open_input_dir = gr.Button(i18n("Open folder"))\n            with gr.Row():\n                with gr.Tabs():\n                    with gr.TabItem(label=i18n("Output folder path")):\n                        store_dir = gr.Textbox(label=i18n("Output folder path"), info=i18n("Audio files will be saved in this folder."), value="results/", interactive=True, scale=4)\n                        with gr.Row():\n                            select_store_btn = gr.Button(i18n("Select folder"))\n                            open_store_btn = gr.Button(i18n("Open folder"))\n        inference_audio = gr.Button(i18n("Input audio files to inference"), variant="primary", visible=True)\n        inference_folder = gr.Button(i18n("Input folder to inference"), variant="primary", visible=False)\n        output_message = gr.Textbox(label=i18n("Output message"), interactive=False)\n        stop_inference = gr.Button(i18n("Forced stop inference"), variant="stop")\n        quit_app = gr.Button(i18n("Close MedleyVox program"), variant="secondary")\n        gr.Markdown(\'\'\'<div align="center">WebUI created by <a href="https://github.com/SUC-DriverOld">Sucial</a>: <a href="https://github.com/SUC-DriverOld/MedleyVox-Inference-WebUI">Githubl</a> | <a href="https://github.com/SUC-DriverOld/MedleyVox-Inference-WebUI/blob/master/LICENSE">LICENSE</a></div>\'\'\')\n\n        audio_tab.select(fn=change_to_audio_infer, outputs=[inference_audio, inference_folder])\n        folder_tab.select(fn=change_to_folder_infer, outputs=[inference_audio, inference_folder])\n        select_input_dir.click(select_folder, outputs=[folder_input])\n        open_input_dir.click(open_folder, inputs=[folder_input])\n        select_store_btn.click(select_folder, outputs=[store_dir])\n        open_store_btn.click(open_folder, inputs=[store_dir])\n        inference_audio.click(inference_audio_fn, inputs=[model_name, use_overlapadd, use_gpu, separate_storage, output_format, vad_method, spectral_features, ola_window_len, ola_hop_len, w2v_nth_layer_output, use_ema_model, mix_consistent_out, reorder_chunks, skip_error, audio_input, store_dir], outputs=[output_message])\n        inference_folder.click(inference_folder_fn, inputs=[model_name, use_overlapadd, use_gpu, separate_storage, output_format, vad_method, spectral_features, ola_window_len, ola_hop_len, w2v_nth_layer_output, use_ema_model, mix_consistent_out, reorder_chunks, skip_error, folder_input, store_dir], outputs=[output_message])\n        stop_inference.click(stop_inference_fn, inputs=[], outputs=[output_message])\n        quit_app.click(quit_app_fn, inputs=[], outputs=[output_message])\n    return demo\n\nif __name__ == "__main__":\n    import argparse\n    import multiprocessing\n    multiprocessing.set_start_method(\'spawn\', force=True)\n\n    parser = argparse.ArgumentParser(description="WebUI for MedleyVox", formatter_class=lambda prog: argparse.RawTextHelpFormatter(prog, max_help_position=60))\n    parser.add_argument("-i", "--ip_address", type=str, default=None, help="IP address to run the server")\n    parser.add_argument("-p", "--port", type=int, default=None, help="Port to run the server")\n    parser.add_argument("-s", "--share", action="store_true", help="Enable gradio share link")\n    parser.add_argument("--auto_clean_cache", action="store_true", help="Auto clean WebUI cache")\n    args = parser.parse_args()\n\n    # เปิดแบบไม่มีหน้าต่าง cmd (pythonw): ไม่มีที่ให้พิมพ์ข้อความ -> เก็บลงไฟล์ cache/webui.log แทน\n    os.makedirs("cache", exist_ok=True)\n    if sys.stdout is None or sys.stderr is None:\n        _log = open(os.path.join("cache", "webui.log"), "w", encoding="utf-8", buffering=1)\n        sys.stdout = sys.stdout or _log\n        sys.stderr = sys.stderr or _log\n\n    # ถ้าเปิดค้างไว้อยู่แล้ว ไม่ต้องเปิดซ้ำ แค่เปิดเบราว์เซอร์ไปหน้าเดิม\n    if args.port and not args.share:\n        import socket, webbrowser\n        with socket.socket() as sock:\n            sock.settimeout(0.5)\n            if sock.connect_ex(("127.0.0.1", args.port)) == 0:\n                webbrowser.open(f"http://127.0.0.1:{args.port}")\n                sys.exit(0)\n\n    os.makedirs("inputs", exist_ok=True)\n    os.makedirs("results", exist_ok=True)\n    os.makedirs("cache", exist_ok=True)\n\n    if args.auto_clean_cache:\n        for item in os.listdir("cache"):\n            if item != "webui.log":\n                path = os.path.join("cache", item)\n                shutil.rmtree(path, ignore_errors=True) if os.path.isdir(path) else os.remove(path)\n\n    os.environ["GRADIO_TEMP_DIR"] = os.path.abspath("cache/")\n    os.environ["HF_HOME"] = os.path.abspath(PRETRAINED_MODEL_DIR)\n\n    webui().queue().launch(inbrowser=True, server_name=args.ip_address, server_port=args.port, share=args.share)', 'inference.py': 'import os\nimport sys\nsys.path.append(os.path.dirname(os.path.abspath(__file__)))\nimport json\nimport argparse\nimport numpy as np\nimport soundfile as sf\nimport librosa\nimport torch\nimport pyloudnorm as pyln\nimport tqdm\nimport traceback\n\nfrom models import load_model_with_args\nfrom functions import load_ola_func_with_args\nfrom utils import loudnorm, str2bool, db2linear\n\nimport warnings\nwarnings.filterwarnings("ignore")\n\nmodel = None\ncontinuous_nnet = None\ndevice = torch.device("cpu")\n\n# ---- ความคืบหน้า: จดลงไฟล์ให้หน้าเว็บอ่าน (นับจำนวนท่อนที่โมเดลแยกเสร็จ) ----\nimport time\nprogress = {"stage": "loading", "file": "", "file_idx": 0, "n_files": 0, "done": 0, "total": None, "start": time.time(), "file_start": time.time()}\n_last_write = [0.0]\n\ndef write_progress(force=False):\n    path = os.environ.get("MEDLEYVOX_PROGRESS")\n    if not path or (not force and time.time() - _last_write[0] < 0.5):\n        return\n    _last_write[0] = time.time()\n    try:\n        with open(path + ".tmp", "w", encoding="utf-8") as f:\n            json.dump(progress, f, ensure_ascii=False)\n        os.replace(path + ".tmp", path)\n    except Exception:\n        pass\n\ndef count_chunks(args, n_samples):\n    # จำนวนท่อนที่ต้องแยกต่อ 1 ช่องเสียง (รู้ล่วงหน้าได้เฉพาะ None / ola / ola_norm / w2v)\n    if not args.use_overlapadd:\n        return 1\n    if args.use_overlapadd not in ("ola", "ola_norm", "w2v"):\n        return None\n    win = int(args.ola_window_len * args.sample_rate) if args.ola_window_len else int(args.seq_dur * args.sample_rate)\n    hop = int(args.ola_hop_len * args.sample_rate) if args.ola_hop_len else int(args.seq_dur * args.sample_rate // 4)\n    # เหมือนที่ asteroid หั่น: เติมขอบหัวท้ายข้างละ 1 ท่อน แล้วเลื่อนทีละ hop\n    return (n_samples + win) // hop + 1\n\n\ndef main():\n    global model, continuous_nnet, device\n\n    parser = argparse.ArgumentParser(description="Inference for MedleyVox", formatter_class=lambda prog: argparse.RawTextHelpFormatter(prog, max_help_position=60))\n    parser.add_argument("--target", type=str, default="vocals")\n    parser.add_argument("--exp_name", type=str, default=None)\n    parser.add_argument("--model_dir", type=str, default="checkpoints", help="model directory")\n    parser.add_argument("--use_gpu", type=str2bool, default=True)\n    parser.add_argument("--use_overlapadd", type=str, default=None, choices=[None, "ola", "ola_norm", "w2v", "w2v_chunk", "sf_chunk"], help="use overlapadd functions, ola, ola_norm, w2v will work with ola_window_len, ola_hop_len argugments. w2v_chunk and sf_chunk is chunk-wise processing based on VAD, so you have to specify the vad_method args. If you use sf_chunk (spectral_featrues_chunk), you also need to specify spectral_features.")\n    parser.add_argument("--vad_method", type=str, default="spec", choices=["spec", "webrtc"], help="what method do you want to use for \'voice activity detection (vad) -- split chunks -- processing. Only valid when \'w2v_chunk\' or \'sf_chunk\' for args.use_overlapadd.")\n    parser.add_argument("--spectral_features", type=str, default="mfcc", choices=["mfcc", "spectral_centroid"], help="what spectral feature do you want to use in correlation calc in speaker assignment (only valid when using sf_chunk)")\n    parser.add_argument("--w2v_ckpt_dir", type=str, default="pretrained", help="only valid when use_overlapadd is \'w2v\' or \'w2v_chunk\'.")\n    parser.add_argument("--w2v_nth_layer_output", nargs="+", type=int, default=[0], help="wav2vec nth layer output")\n    parser.add_argument("--ola_window_len", type=float, default=None, help="ola window size in [sec]")\n    parser.add_argument("--ola_hop_len", type=float, default=None, help="ola hop size in [sec]")\n    parser.add_argument("--use_ema_model", type=str2bool, default=True, help="use ema model or online model? only vaind when args.ema it True (model trained with ema)")\n    parser.add_argument("--mix_consistent_out", type=str2bool, default=True, help="only valid when the model is trained with mixture_consistency loss. Default is True.")\n    parser.add_argument("--reorder_chunks", type=str2bool, default=True, help="ola reorder chunks")\n    parser.add_argument("--inference_data_dir", type=str, default="inputs", help="data where you want to separate")\n    parser.add_argument("--results_save_dir", type=str, default="results", help="save directory")\n    parser.add_argument("--output_format", type=str, default="wav", choices=["wav", "mp3", "flac"], help="output format")\n    parser.add_argument("--separate_storage", type=str2bool, default=False, help="save results in separate folders with the same name as the input file")\n    parser.add_argument("--skip_error", type=str2bool, default=True, help="skip error files while separating instead of stopping")\n    args, _ = parser.parse_known_args()\n\n    args.exp_result_dir = os.path.join(args.model_dir, args.exp_name)\n    with open(os.path.join(args.exp_result_dir, f"{args.target}.json"), "r") as f:\n        args_dict = json.load(f)\n    for key, value in args_dict["args"].items():\n        setattr(args, key, value)\n\n    if args.use_overlapadd == "w2v" or args.use_overlapadd == "w2v_chunk":\n        assert os.path.exists(os.path.join(args.w2v_ckpt_dir, "xlsr_53_56k.pt")), "Please download the pretrained model xlsr_53_56k.pt from https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt and put it in the \'pretrained\' folder!"\n\n    # load model architecture\n    model = load_model_with_args(args)\n\n    if torch.cuda.is_available() and args.use_gpu:\n        device = torch.device("cuda")\n        print("Using GPU for inference")\n    else:\n        print("Using CPU for inference")\n\n    target_model_path = f"{args.exp_result_dir}/{args.target}.pth"\n    checkpoint = torch.load(target_model_path, map_location=device)\n    if args.ema and args.use_ema_model:\n        print("Use ema model")\n        model_dict = model.state_dict()\n        # 1. filter out unnecessary keys\n        checkpoint = {\n            k.replace("ema_model.module.", ""): v\n            for k, v in checkpoint.items()\n            if k.replace("ema_model.module.", "") in model_dict\n        }\n        # 2. overwrite entries in the existing state dict\n        model_dict.update(checkpoint)\n        # 3. load the new state dict\n        model.load_state_dict(model_dict)\n    elif args.ema and not args.use_ema_model:\n        print("Use ema online model")\n        model_dict = model.state_dict()\n        # 1. filter out unnecessary keys\n        checkpoint = {\n            k.replace("online_model.module.", ""): v\n            for k, v in checkpoint.items()\n            if k.replace("online_model.module.", "") in model_dict\n        }\n        # 2. overwrite entries in the existing state dict\n        model_dict.update(checkpoint)\n        # 3. load the new state dict\n        model.load_state_dict(model_dict)\n    else:\n        model.load_state_dict(checkpoint)\n\n    model.eval()\n    meter = pyln.Meter(args.sample_rate)\n\n    _forward = model.forward\n    def counted_forward(*a, **k):\n        out = _forward(*a, **k)\n        progress["done"] += 1\n        write_progress()\n        return out\n    model.forward = counted_forward\n\n    if args.use_overlapadd:\n        continuous_nnet = load_ola_func_with_args(args, model, device, meter)\n\n    data_list = os.listdir(args.inference_data_dir)\n    print(f"Total files found: {len(data_list)}")\n    progress["n_files"] = len(data_list)\n    data_list = tqdm.tqdm(data_list, desc="Separating")\n\n    success_files = []\n    error_files = []\n    for idx, data_path in enumerate(data_list):\n        song_name = os.path.basename(data_path).split(".")[0]\n        progress.update(stage="reading", file=data_path, file_idx=idx + 1, done=0, total=None, file_start=time.time())\n        write_progress(force=True)\n        data_list.set_postfix_str(song_name)\n\n        if args.separate_storage:\n            save_dir = os.path.join(args.results_save_dir, song_name)\n        else:\n            save_dir = args.results_save_dir\n        os.makedirs(save_dir, exist_ok=True)\n\n        try:\n            mixture, _ = librosa.load(\n                path=os.path.join(args.inference_data_dir, data_path),\n                sr=args.sample_rate,\n                mono=False,\n                dtype=np.float32\n            )\n        except Exception as e:\n            print(f"Error loading {data_path}, error: {e}, skipping...")\n            traceback.print_exc()\n            error_files.append(data_path)\n            continue\n\n        try:\n            if len(mixture.shape) != 1 and mixture.shape[0] > 2:\n                mixture = np.mean(mixture, axis=0)\n\n            per_channel = count_chunks(args, mixture.shape[-1])\n            channels = 2 if len(mixture.shape) == 2 else 1\n            progress.update(stage="separating", done=0, total=per_channel * channels if per_channel else None, file_start=time.time())\n            write_progress(force=True)\n\n            if len(mixture.shape) == 2:\n                left = mixture[0, :]\n                right = mixture[1, :]\n                left_out_wav_1, left_out_wav_2 = inference(args, left, meter)\n                right_out_wav_1, right_out_wav_2 = inference(args, right, meter)\n                out_wav_1 = np.stack([left_out_wav_1, right_out_wav_1], axis=-1)\n                out_wav_2 = np.stack([left_out_wav_2, right_out_wav_2], axis=-1)\n            else:\n                out_wav_1, out_wav_2 = inference(args, mixture, meter)\n\n            save_wav_path_1 = os.path.join(save_dir, f"{song_name}_output_1.{args.output_format}")\n            save_wav_path_2 = os.path.join(save_dir, f"{song_name}_output_2.{args.output_format}")\n\n            progress["stage"] = "saving"\n            write_progress(force=True)\n            sf.write(save_wav_path_1, out_wav_1, args.sample_rate)\n            sf.write(save_wav_path_2, out_wav_2, args.sample_rate)\n            success_files.append(data_path)\n        except Exception as e:\n            if args.skip_error:\n                print(f"Error separating {data_path}, error: {e}, skipping...")\n                traceback.print_exc()\n                error_files.append(data_path)\n                continue\n            else:\n                raise e\n\n    progress.update(stage="finished", success=len(success_files), errors=error_files)\n    write_progress(force=True)\n    print("Separation done!")\n    if len(success_files) == 0:\n        print("No files separated")\n    else:\n        print(f"Successfully separated {len(success_files)} files: {success_files}")\n    if len(error_files) == 0:\n        print("No files with errors")\n    else:\n        print(f"Error in separating {len(error_files)} files: {error_files}")\n\ndef inference(args, mix, meter):\n    assert (model is not None or continuous_nnet is not None), "model is not loaded"\n\n    mix, adjusted_gain = loudnorm(mix, -24.0, meter)\n    mix = np.expand_dims(mix, axis=0)\n    mix = mix.reshape(1, mix.shape[0], mix.shape[1])\n    mix = torch.as_tensor(mix, dtype=torch.float32).to(device)\n\n    if args.use_overlapadd:\n        out_wavs = continuous_nnet.forward(mix)\n    else:\n        out_wavs = model.separate(mix)\n\n    if args.use_gpu:\n        out_wav_1 = out_wavs[0, 0, :].cpu().detach().numpy()\n        out_wav_2 = out_wavs[0, 1, :].cpu().detach().numpy()\n    else:\n        out_wav_1 = out_wavs[0, 0, :]\n        out_wav_2 = out_wavs[0, 1, :]\n\n    out_wav_1 = out_wav_1 * db2linear(-adjusted_gain)\n    out_wav_2 = out_wav_2 * db2linear(-adjusted_gain)\n\n    return out_wav_1, out_wav_2\n\n\nif __name__ == "__main__":\n    main()\n'}
for file_name, code in patched_files.items():
    (repo_path / file_name).write_text(code, encoding='utf-8')
os.environ['LANGUAGE'] = language

clear_output()
print('เปิดหน้าเว็บ... รอจนขึ้นลิงก์ https://xxxx.gradio.live แล้วกดลิงก์นั้น')
print('ไฟล์เสียง: อัปโหลดในหน้าเว็บได้เลย ผลลัพธ์อยู่ในโฟลเดอร์ results (ดูได้จากแถบไฟล์ซ้ายมือ)')
!{py} webui.py --share
